In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
import boto3
import os
from tqdm import tqdm
import s3fs
import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## Helper Classes

In [ ]:
class TimeSeriesPreprocessor:
    """Preprocessing and stationarity testing for time series."""
    
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.fs = s3fs.S3FileSystem()
        self.s3_client = boto3.client('s3')
        self.df_data = None
        self.series_company = None
        self.series_original = None
        self.series_differenced = None
        self.df_train = None
        self.df_test = None
    
    def import_data(self, str_filename='employee_attrition_data.csv'):
        """Load raw data from S3."""
        str_uri = f's3://{self.str_bucket}/00_data_collection/{str_filename}'
        self.df_data = pd.read_csv(str_uri)
        self.df_data['date'] = pd.to_datetime(self.df_data['date'])
        self.df_data = self.df_data.sort_values('date').reset_index(drop=True)
        print(f'Loaded {len(self.df_data)} records from S3')
        return self.df_data
    
    def aggregate_company_level(self):
        """Aggregate to company-wide monthly attrition."""
        df_company = self.df_data.groupby('date').agg({
            'headcount': 'sum',
            'new_hires': 'sum',
            'departures': 'sum',
            'attrition_rate': 'mean',
            'avg_tenure_months': 'mean',
            'avg_satisfaction_score': 'mean'
        }).reset_index()
        
        self.series_company = df_company['attrition_rate'].values
        self.series_original = df_company['attrition_rate'].copy()
        
        print(f'Aggregated to {len(df_company)} company-level records')
        return df_company
    
    def test_stationarity(self, series=None):
        """Perform ADF and KPSS tests."""
        if series is None:
            series = self.series_company
        
        print('\n=== Stationarity Tests ===')
        
        # ADF Test
        result_adf = adfuller(series, autolag='AIC')
        print('\nAugmented Dickey-Fuller Test:')
        print(f'  Test Statistic: {result_adf[0]:.6f}')
        print(f'  P-Value: {result_adf[1]:.6f}')
        print(f'  Critical Values:')
        for key, value in result_adf[4].items():
            print(f'    {key}: {value:.3f}')
        
        bool_adf_stationary = result_adf[1] < 0.05
        print(f'  Result: {"STATIONARY" if bool_adf_stationary else "NON-STATIONARY"}')
        
        # KPSS Test
        result_kpss = kpss(series, regression='c', nlags='auto')
        print('\nKPSS Test:')
        print(f'  Test Statistic: {result_kpss[0]:.6f}')
        print(f'  P-Value: {result_kpss[1]:.6f}')
        print(f'  Critical Values:')
        for key, value in result_kpss[3].items():
            print(f'    {key}: {value:.3f}')
        
        bool_kpss_stationary = result_kpss[1] > 0.05
        print(f'  Result: {"STATIONARY" if bool_kpss_stationary else "NON-STATIONARY"}')
        
        return bool_adf_stationary, bool_kpss_stationary
    
    def make_stationary(self):
        """Apply differencing if needed."""
        series = self.series_company.copy()
        int_diff_order = 0
        
        # First differencing
        series_diff1 = np.diff(series, n=1)
        bool_adf_d1, bool_kpss_d1 = self.test_stationarity(series_diff1)
        
        if bool_adf_d1 and bool_kpss_d1:
            self.series_differenced = series_diff1
            int_diff_order = 1
            print(f'\nFirst differencing makes series stationary (d=1)')
        else:
            # Try seasonal differencing (lag=12)
            series_diff12 = np.diff(series, n=12)
            bool_adf_d12, bool_kpss_d12 = self.test_stationarity(series_diff12)
            
            if bool_adf_d12 and bool_kpss_d12:
                self.series_differenced = series_diff12
                int_diff_order = 12
                print(f'\nSeasonal differencing (lag=12) makes series stationary')
            else:
                self.series_differenced = series_diff1
                int_diff_order = 1
                print(f'\nUsing first differencing (d=1)')
        
        # Plot differencing
        fig, axes = plt.subplots(2, 1, figsize=(14, 8))
        
        axes[0].plot(self.series_company, linewidth=1.5, label='Original')
        axes[0].set_title('Original Time Series', fontsize=12, fontweight='bold')
        axes[0].set_ylabel('Attrition Rate')
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()
        
        axes[1].plot(self.series_differenced, linewidth=1.5, label=f'Differenced (order={int_diff_order})', color='#ff7f0e')
        axes[1].set_title(f'Differenced Time Series (d={int_diff_order})', fontsize=12, fontweight='bold')
        axes[1].set_ylabel('Differenced Attrition Rate')
        axes[1].set_xlabel('Time Index')
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()
        
        plt.tight_layout()
        str_path = f'{self.str_dirname_output}/09_differencing.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved differencing plot to {str_path}')
        plt.show()
        
        return int_diff_order
    
    def create_features(self, df_company):
        """Engineer lag features and rolling averages."""
        df_features = df_company.copy()
        
        # Lag features
        for int_lag in [1, 3, 6, 12]:
            df_features[f'attrition_lag_{int_lag}'] = df_features['attrition_rate'].shift(int_lag)
        
        # Rolling averages
        for int_window in [3, 6, 12]:
            df_features[f'attrition_rolling_mean_{int_window}'] = df_features['attrition_rate'].rolling(int_window).mean()
        
        # Month dummies
        df_features['month'] = pd.to_datetime(df_features['date']).dt.month
        for int_month in range(1, 13):
            df_features[f'month_{int_month}'] = (df_features['month'] == int_month).astype(int)
        
        # Drop rows with NaN from lagging
        df_features = df_features.dropna().reset_index(drop=True)
        
        print(f'Created features: {df_features.shape[1]} columns')
        print(f'Final dataset: {df_features.shape[0]} rows (after lag handling)')
        
        return df_features
    
    def split_data(self, df_company, int_test_months=12):
        """Temporal train/test split."""
        int_split_idx = len(df_company) - int_test_months
        
        self.df_train = df_company.iloc[:int_split_idx].copy()
        self.df_test = df_company.iloc[int_split_idx:].copy()
        
        print(f'\nTrain set: {len(self.df_train)} samples ({self.df_train["date"].min()} to {self.df_train["date"].max()})')
        print(f'Test set: {len(self.df_test)} samples ({self.df_test["date"].min()} to {self.df_test["date"].max()})')
        
        # Visualize split
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.plot(self.df_train['date'], self.df_train['attrition_rate'], linewidth=2, label='Train', marker='o')
        ax.plot(self.df_test['date'], self.df_test['attrition_rate'], linewidth=2, label='Test', marker='s', color='#d62728')
        ax.axvline(x=self.df_train['date'].iloc[-1], color='black', linestyle='--', linewidth=2, alpha=0.7)
        ax.set_xlabel('Date', fontsize=11)
        ax.set_ylabel('Attrition Rate', fontsize=11)
        ax.set_title('Train/Test Split', fontsize=13, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        str_path = f'{self.str_dirname_output}/10_train_test_split.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved split visualization to {str_path}')
        plt.show()
        
        return self.df_train, self.df_test
    
    def save_splits(self, str_bucket):
        """Save train/test splits to S3."""
        # Save to local first
        str_train_path = f'{self.str_dirname_output}/train_data.csv'
        str_test_path = f'{self.str_dirname_output}/test_data.csv'
        
        self.df_train.to_csv(str_train_path, index=False)
        self.df_test.to_csv(str_test_path, index=False)
        
        # Upload to S3
        try:
            self.s3_client.upload_file(
                str_train_path,
                str_bucket,
                '02_preprocessing/train_data.csv'
            )
            self.s3_client.upload_file(
                str_test_path,
                str_bucket,
                '02_preprocessing/test_data.csv'
            )
            print(f'\nUploaded splits to S3:')
            print(f'  s3://{str_bucket}/02_preprocessing/train_data.csv')
            print(f'  s3://{str_bucket}/02_preprocessing/test_data.csv')
        except Exception as e:
            print(f'Error uploading to S3: {e}')

## Constants

In [ ]:
str_bucket = 'time-series-forecasting-demo-repo'
str_task = 'employee_attrition_forecasting'
str_dirname_output = './output'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

print(f'Output directory: {str_dirname_output}')

## Load and Aggregate Data

In [ ]:
preprocessor = TimeSeriesPreprocessor(str_bucket, str_dirname_output)
df_raw = preprocessor.import_data()
df_company = preprocessor.aggregate_company_level()

print(f'\nAggregated Data Summary:')
print(df_company.describe())

## Stationarity Testing

In [ ]:
print('Testing original series:')
bool_adf_orig, bool_kpss_orig = preprocessor.test_stationarity()

## Differencing

In [ ]:
int_d = preprocessor.make_stationary()

## Feature Engineering

In [ ]:
df_features = preprocessor.create_features(df_company)
print(f'\nFeature Summary:')
print(df_features.head(10))

## Train/Test Split

In [ ]:
df_train, df_test = preprocessor.split_data(df_company, int_test_months=12)

## Save Splits to S3

In [ ]:
preprocessor.save_splits(str_bucket)

## Preprocessing Summary

In [ ]:
print('\n=== PREPROCESSING SUMMARY ===')
print(f'Stationarity:')
print(f'  Original series (ADF): {"Stationary" if bool_adf_orig else "Non-stationary"}')
print(f'  Original series (KPSS): {"Stationary" if bool_kpss_orig else "Non-stationary"}')
print(f'  Differencing order (d): {int_d}')
print(f'\nData Splits:')
print(f'  Train: {len(df_train)} samples')
print(f'  Test: {len(df_test)} samples')
print(f'\nReady for modeling!')